# 05 - Model Tuning

The objective of this notebook is to optimize the hyperparameters of the candidate
fraud detection models identified during model selection.

The models will be tuned using cross-validation on the training data only.
The test set will remain untouched until the final evaluation stage.

Primary optimization metric:
- PR-AUC (Average Precision)

Secondary metrics:
- ROC-AUC
- Precision
- Recall
- F1-score

In [3]:
import os
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [4]:
from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

In [5]:
from sklearn.metrics import average_precision_score

## 2. Load Training Data

For hyperparameter tuning, we use the original training set before SMOTE.

SMOTE will be applied separately within each cross-validation training fold
to prevent information leakage between the training and validation folds.

The test set remains completely untouched until final model evaluation.

In [6]:
# Load the original training data
X_train_processed = pd.read_csv(
    "../data/Processed/X_train_processed.csv"
)

X_test_processed = pd.read_csv(
    "../data/Processed/X_test_processed.csv"
)

print("X_train shape:", X_train_processed.shape)
#print("y_train shape:", y_train.shape)

df = pd.read_csv(
    "../data/Processed/creditcard_clean.csv"
)

from sklearn.model_selection import train_test_split

X_full = df.drop(columns=["Class"])
y_full = df["Class"]

X_train_full, X_test_full, y_train_original, y_test_original = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=42
)

y_test = pd.read_csv(
    "../data/Processed/y_test.csv"
).squeeze()
print("y_train shape:", y_train_original.shape)
X_train=X_train_processed
y_train=y_train_original
X_test=X_test_processed


X_train shape: (226980, 32)
y_train shape: (226980,)


In [7]:
print(y_train.value_counts())

Class
0    226602
1       378
Name: count, dtype: int64


In [8]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [10]:
smote = SMOTE(
    random_state=RANDOM_STATE
)

In [11]:
logistic_pipeline = Pipeline([
    ("smote", smote),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

In [12]:
print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [13]:
print("Class in X_train:", "Class" in X_train.columns)
print("Number of features:", X_train.shape[1])

print("Fraud cases:", (y_train == 1).sum())
print("Legitimate cases:", (y_train == 0).sum())
print("Fraud rate:", y_train.mean())

Class in X_train: False
Number of features: 32
Fraud cases: 378
Legitimate cases: 226602
Fraud rate: 0.0016653449643140364


In [14]:
logistic_param_dist = {
    "model__C": np.logspace(-3, 2, 20),
    "model__class_weight": [None, "balanced"]
}

In [15]:
logistic_search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=logistic_param_dist,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

In [16]:
logistic_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'model__C': array([1.0000...00000000e+02]), 'model__class_weight': [None, 'balanced']}"
,n_iter,20
,scoring,'average_precision'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [17]:
print("Best CV PR-AUC:", logistic_search.best_score_)
print()
print("Best parameters:")
print(logistic_search.best_params_)

Best CV PR-AUC: 0.7557280951786305

Best parameters:
{'model__class_weight': 'balanced', 'model__C': np.float64(0.011288378916846888)}


In [18]:
logistic_best_model = logistic_search.best_estimator_

print(logistic_best_model)

Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('model',
                 LogisticRegression(C=np.float64(0.011288378916846888),
                                    class_weight='balanced', max_iter=1000,
                                    random_state=42))])


In [19]:
logistic_results = pd.DataFrame(
    logistic_search.cv_results_
)

logistic_results[
    [
        "mean_test_score",
        "std_test_score",
        "param_model__C",
        "param_model__class_weight"
    ]
].sort_values(
    "mean_test_score",
    ascending=False
).head(10)

,mean_test_score,std_test_score,param_model__C,param_model__class_weight
15,0.755728,0.024905,0.011288,None
11,0.755728,0.024905,0.011288,balanced
5,0.755559,0.025309,0.037927,None
12,0.755559,0.025309,0.037927,balanced
16,0.755391,0.025539,0.127427,balanced
1,0.755391,0.025539,0.127427,None
0,0.755359,0.025191,0.233572,balanced
19,0.755341,0.025128,16.237767,balanced
14,0.755341,0.025128,29.763514,None
8,0.755341,0.025128,100.000000,balanced


In [20]:
rf_pipeline = Pipeline([
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [21]:
rf_param_dist = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", 0.5],
    "model__class_weight": [None, "balanced"]
}

In [22]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=10,
    scoring="average_precision",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

In [23]:
print(rf_search)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('smote', SMOTE(random_state=42)),
                                             ('model',
                                              RandomForestClassifier(n_jobs=-1,
                                                                     random_state=42))]),
                   n_jobs=-1,
                   param_distributions={'model__class_weight': [None,
                                                                'balanced'],
                                        'model__max_depth': [None, 10, 20],
                                        'model__max_features': ['sqrt', 0.5],
                                        'model__min_samples_leaf': [1, 2, 4],
                                        'model__min_samples_split': [2, 5, 10],
                                        'model__n_estimators': [200, 300]},
                   random_state=42, scoring='average_precision', verbose=1)


In [ ]:
rf_search.fit(X_train, y_train)

In [ ]:
print("Best CV PR-AUC:", rf_search.best_score_)
print()
print("Best parameters:")
print(rf_search.best_params_)

In [24]:
xgb_pipeline = Pipeline([
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

In [25]:
xgb_param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

In [26]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=5,
    scoring="average_precision",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

In [27]:
xgb_search.fit(X_train, y_train)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_distributions,"{'model__colsample_bytree': [0.8, 1.0], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [3, 5, ...], 'model__n_estimators': [100, 200, ...], ...}"
,n_iter,5
,scoring,'average_precision'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [28]:
print("Best CV PR-AUC:", xgb_search.best_score_)
print()

print("Best parameters:")
print(xgb_search.best_params_)

Best CV PR-AUC: 0.8501176962245162

Best parameters:
{'model__subsample': 0.8, 'model__n_estimators': 300, 'model__max_depth': 5, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}


In [29]:
xgb_results = pd.DataFrame(xgb_search.cv_results_)

xgb_results[
    [
        "mean_test_score",
        "std_test_score",
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__learning_rate",
        "param_model__subsample",
        "param_model__colsample_bytree"
    ]
].sort_values(
    "mean_test_score",
    ascending=False
).head(5)

,mean_test_score,std_test_score,param_model__n_estimators,param_model__max_depth,param_model__learning_rate,param_model__subsample,param_model__colsample_bytree
4,0.850118,0.039342,300,5,0.10,0.8,0.8
1,0.841484,0.039946,200,5,0.10,0.8,1.0
0,0.814453,0.039523,300,3,0.05,0.8,0.8
3,0.744994,0.015360,100,3,0.05,0.8,0.8
2,0.743392,0.026348,100,3,0.10,0.8,0.8


In [30]:
tuning_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "CV_PR_AUC": [
        logistic_search.best_score_,
        np.nan,
        xgb_search.best_score_
    ],
    "Status": [
        "Tuned",
        "Baseline retained",
        "Tuned"
    ]
})

tuning_summary

,Model,CV_PR_AUC,Status
0,Logistic Regression,0.755728,Tuned
1,Random Forest,NaN,Baseline retained
2,XGBoost,0.850118,Tuned


In [31]:
final_xgb_pipeline = Pipeline([
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

In [32]:
final_xgb_pipeline.fit(X_train, y_train)

,steps,"[('smote', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,sampling_strategy,'auto'
,random_state,42
,k_neighbors,5
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None


In [33]:
y_test_proba = final_xgb_pipeline.predict_proba(X_test)[:, 1]

In [34]:
y_test_pred = (y_test_proba >= 0.5).astype(int)

In [35]:
test_pr_auc = average_precision_score(
    y_test,
    y_test_proba
)

print("Test PR-AUC:", test_pr_auc)

Test PR-AUC: 0.8135294053352436


In [36]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [37]:
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, y_test_proba)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", roc_auc)

Precision: 0.8588235294117647
Recall: 0.7684210526315789
F1: 0.8111111111111111
ROC-AUC: 0.9721379935691198


In [38]:
cm = confusion_matrix(y_test, y_test_pred)

print(cm)

[[56639    12]
 [   22    73]]


In [39]:
from sklearn.model_selection import cross_val_predict

In [40]:
oof_proba = cross_val_predict(
    final_xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

In [41]:
oof_proba

array([2.0311372e-06, 2.0240639e-06, 6.2029554e-05, ..., 1.8702331e-05,
       1.6451639e-05, 1.3367656e-06], shape=(226980,), dtype=float32)

In [42]:
print("OOF predictions:", len(oof_proba))
print("Training samples:", len(y_train))

OOF predictions: 226980
Training samples: 226980


In [46]:
oof_pr_auc = average_precision_score(
    y_train,
    oof_proba
)

print(f"OOF PR-AUC:0:{ oof_pr_auc:.4f}")

OOF PR-AUC:0:0.8437


In [44]:
thresholds = np.arange(
    0.10,
    0.91,
    0.05
)

In [45]:
threshold_results = []

for threshold in thresholds:

    y_pred = (oof_proba >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, y_pred, zero_division=0),
        "Recall": recall_score(y_train, y_pred, zero_division=0),
        "F1": f1_score(y_train, y_pred, zero_division=0),
        "FP": ((y_train == 0) & (y_pred == 1)).sum(),
        "FN": ((y_train == 1) & (y_pred == 0)).sum(),
        "TP": ((y_train == 1) & (y_pred == 1)).sum(),
        "TN": ((y_train == 0) & (y_pred == 0)).sum()
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results

,Threshold,Precision,Recall,F1,FP,FN,TP,TN
0,0.10,0.647541,0.835979,0.729792,172,62,316,226430
1,0.15,0.719178,0.833333,0.772059,123,63,315,226479
2,0.20,0.783042,0.830688,0.806162,87,64,314,226515
3,0.25,0.805128,0.830688,0.817708,76,64,314,226526
4,0.30,0.821990,0.830688,0.826316,68,64,314,226534
5,0.35,0.840970,0.825397,0.833111,59,66,312,226543
6,0.40,0.861878,0.825397,0.843243,50,66,312,226552
7,0.45,0.878873,0.825397,0.851296,43,66,312,226559
8,0.50,0.886040,0.822751,0.853224,40,67,311,226562
9,0.55,0.896254,0.822751,0.857931,36,67,311,226566


In [47]:
best_f1_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

best_f1_row

Threshold         0.800000
Precision         0.938650
Recall            0.809524
F1                0.869318
FP               20.000000
FN               72.000000
TP              306.000000
TN           226582.000000
Name: 14, dtype: float64

In [48]:
test_threshold = 0.80

y_test_pred_080 = (
    y_test_proba >= test_threshold
).astype(int)

In [49]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

precision_080 = precision_score(
    y_test,
    y_test_pred_080,
    zero_division=0
)

recall_080 = recall_score(
    y_test,
    y_test_pred_080,
    zero_division=0
)

f1_080 = f1_score(
    y_test,
    y_test_pred_080,
    zero_division=0
)

roc_auc_080 = roc_auc_score(
    y_test,
    y_test_proba
)

pr_auc_080 = average_precision_score(
    y_test,
    y_test_proba
)

cm_080 = confusion_matrix(
    y_test,
    y_test_pred_080
)

print(f"Precision: {precision_080:.4f}")
print(f"Recall:    {recall_080:.4f}")
print(f"F1:        {f1_080:.4f}")
print(f"ROC-AUC:   {roc_auc_080:.4f}")
print(f"PR-AUC:    {pr_auc_080:.4f}")
print("\nConfusion Matrix:")
print(cm_080)

Precision: 0.9359
Recall:    0.7684
F1:        0.8439
ROC-AUC:   0.9721
PR-AUC:    0.8135

Confusion Matrix:
[[56646     5]
 [   22    73]]


In [50]:
final_comparison = pd.DataFrame([
    {
        "Model": "Baseline XGBoost",
        "Threshold": 0.50,
        "Precision": 0.8488,
        "Recall": 0.7684,
        "F1": 0.8066,
        "ROC-AUC": 0.9691,
        "PR-AUC": 0.8085,
        "FP": 13,
        "FN": 22
    },
    {
        "Model": "Tuned XGBoost",
        "Threshold": 0.50,
        "Precision": 0.8588,
        "Recall": 0.7684,
        "F1": 0.8111,
        "ROC-AUC": 0.9721,
        "PR-AUC": 0.8135,
        "FP": 12,
        "FN": 22
    },
    {
        "Model": "Tuned XGBoost",
        "Threshold": 0.80,
        "Precision": 0.9359,
        "Recall": 0.7684,
        "F1": 0.8439,
        "ROC-AUC": 0.9721,
        "PR-AUC": 0.8135,
        "FP": 5,
        "FN": 22
    }
])

final_comparison

,Model,Threshold,Precision,Recall,F1,ROC-AUC,PR-AUC,FP,FN
0,Baseline XGBoost,0.5,0.8488,0.7684,0.8066,0.9691,0.8085,13,22
1,Tuned XGBoost,0.5,0.8588,0.7684,0.8111,0.9721,0.8135,12,22
2,Tuned XGBoost,0.8,0.9359,0.7684,0.8439,0.9721,0.8135,5,22


In [51]:
import os

models_dir = "../models"

print("Files in models/:")
for file in os.listdir(models_dir):
    print("-", file)

Files in models/:
- preprocessor.joblib


In [52]:
import joblib

model_path = "../models/xgboost_fraud_pipeline.joblib"

joblib.dump(
    final_xgb_pipeline,
    model_path
)

print(f"Model saved to: {model_path}")

Model saved to: ../models/xgboost_fraud_pipeline.joblib


In [53]:
import os

print(os.path.exists(model_path))
print(os.path.getsize(model_path), "bytes")

True
751881 bytes


In [54]:
import json

threshold_path = "../models/threshold.json"

threshold_metadata = {
    "threshold": 0.80,
    "selection_method": "OOF F1 maximization",
    "cv_strategy": "5-fold StratifiedKFold",
    "random_state": RANDOM_STATE
}

with open(threshold_path, "w") as file:
    json.dump(threshold_metadata, file, indent=4)

print(f"Threshold saved to: {threshold_path}")

Threshold saved to: ../models/threshold.json


In [55]:
with open(threshold_path, "r") as file:
    saved_threshold = json.load(file)

print(saved_threshold)

{'threshold': 0.8, 'selection_method': 'OOF F1 maximization', 'cv_strategy': '5-fold StratifiedKFold', 'random_state': 42}


In [56]:
import json

metadata_path = "../models/model_metadata.json"

model_metadata = {
    "model": "XGBoost",
    "model_type": "XGBClassifier",
    "pipeline": [
        "SMOTE",
        "XGBoost"
    ],
    "random_state": RANDOM_STATE,

    "hyperparameters": {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    "tuning": {
        "method": "RandomizedSearchCV",
        "cv_folds": 3,
        "scoring": "average_precision",
        "best_cv_pr_auc": 0.8501176962
    },

    "threshold_selection": {
        "method": "OOF F1 maximization",
        "cv_folds": 5,
        "selected_threshold": 0.80,
        "oof_pr_auc": 0.8437,
        "oof_precision": 0.938650,
        "oof_recall": 0.809524,
        "oof_f1": 0.869318
    },

    "test_evaluation": {
        "threshold": 0.80,
        "precision": 0.9359,
        "recall": 0.7684,
        "f1": 0.8439,
        "roc_auc": 0.9721,
        "pr_auc": 0.8135,
        "true_negatives": 56646,
        "false_positives": 5,
        "false_negatives": 22,
        "true_positives": 73
    }
}

with open(metadata_path, "w") as file:
    json.dump(model_metadata, file, indent=4)

print(f"Metadata saved to: {metadata_path}")

Metadata saved to: ../models/model_metadata.json


In [57]:
with open(metadata_path, "r") as file:
    saved_metadata = json.load(file)

print(json.dumps(saved_metadata, indent=4))

{
    "model": "XGBoost",
    "model_type": "XGBClassifier",
    "pipeline": [
        "SMOTE",
        "XGBoost"
    ],
    "random_state": 42,
    "hyperparameters": {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    "tuning": {
        "method": "RandomizedSearchCV",
        "cv_folds": 3,
        "scoring": "average_precision",
        "best_cv_pr_auc": 0.8501176962
    },
    "threshold_selection": {
        "method": "OOF F1 maximization",
        "cv_folds": 5,
        "selected_threshold": 0.8,
        "oof_pr_auc": 0.8437,
        "oof_precision": 0.93865,
        "oof_recall": 0.809524,
        "oof_f1": 0.869318
    },
    "test_evaluation": {
        "threshold": 0.8,
        "precision": 0.9359,
        "recall": 0.7684,
        "f1": 0.8439,
        "roc_auc": 0.9721,
        "pr_auc": 0.8135,
        "true_negatives": 56646,
        "false_positives": 5,
        "false_

In [58]:
# Load the saved model
loaded_xgb_pipeline = joblib.load(
    "../models/xgboost_fraud_pipeline.joblib"
)

# Generate predictions using the loaded model
loaded_test_proba = loaded_xgb_pipeline.predict_proba(
    X_test
)[:, 1]

# Compare predictions
predictions_match = np.allclose(
    y_test_proba,
    loaded_test_proba
)

print("Predictions match:", predictions_match)

Predictions match: True


In [59]:
# Load the saved threshold
with open("../models/threshold.json", "r") as file:
    saved_threshold_data = json.load(file)

saved_threshold = saved_threshold_data["threshold"]

# Apply the saved threshold
loaded_test_pred = (
    loaded_test_proba >= saved_threshold
).astype(int)

# Calculate confusion matrix
loaded_cm = confusion_matrix(
    y_test,
    loaded_test_pred
)

print("Saved threshold:", saved_threshold)
print("\nConfusion Matrix:")
print(loaded_cm)

Saved threshold: 0.8

Confusion Matrix:
[[56646     5]
 [   22    73]]


### ML CHeck

In [66]:
from pathlib import Path

models_dir = Path("../models")

for path in models_dir.iterdir():
    print(f"{path.name:<35} {path.stat().st_size:,} bytes")

model_metadata.json                 1,094 bytes
preprocessor.joblib                 4,035 bytes
threshold.json                      145 bytes
xgboost_fraud_pipeline.joblib       751,881 bytes


In [64]:
import json

with open("../models/model_metadata.json", "r") as f:
    metadata = json.load(f)

metadata

{'model': 'XGBoost',
 'model_type': 'XGBClassifier',
 'pipeline': ['SMOTE', 'XGBoost'],
 'random_state': 42,
 'hyperparameters': {'n_estimators': 300,
  'max_depth': 5,
  'learning_rate': 0.1,
  'subsample': 0.8,
  'colsample_bytree': 0.8},
 'tuning': {'method': 'RandomizedSearchCV',
  'cv_folds': 3,
  'scoring': 'average_precision',
  'best_cv_pr_auc': 0.8501176962},
 'threshold_selection': {'method': 'OOF F1 maximization',
  'cv_folds': 5,
  'selected_threshold': 0.8,
  'oof_pr_auc': 0.8437,
  'oof_precision': 0.93865,
  'oof_recall': 0.809524,
  'oof_f1': 0.869318},
 'test_evaluation': {'threshold': 0.8,
  'precision': 0.9359,
  'recall': 0.7684,
  'f1': 0.8439,
  'roc_auc': 0.9721,
  'pr_auc': 0.8135,
  'true_negatives': 56646,
  'false_positives': 5,
  'false_negatives': 22,
  'true_positives': 73}}

In [65]:
with open("../models/threshold.json", "r") as f:
    threshold = json.load(f)

threshold

{'threshold': 0.8,
 'selection_method': 'OOF F1 maximization',
 'cv_strategy': '5-fold StratifiedKFold',
 'random_state': 42}

In [68]:
import joblib

pipeline = joblib.load("../models/xgboost_fraud_pipeline.joblib")
preprocessor = joblib.load("../models/preprocessor.joblib")

print("Pipeline type:")
print(type(pipeline))

print("\nPreprocessor type:")
print(type(preprocessor))

Pipeline type:
<class 'imblearn.pipeline.Pipeline'>

Preprocessor type:
<class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [69]:
print(pipeline)

Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                      

In [70]:
print(preprocessor)

ColumnTransformer(transformers=[('amount',
                                 Pipeline(steps=[('log1p',
                                                  FunctionTransformer(feature_names_out='one-to-one',
                                                                      func=<ufunc 'log1p'>)),
                                                 ('scaler', StandardScaler())]),
                                 ['Amount']),
                                ('numeric', StandardScaler(),
                                 ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6',
                                  'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13',
                                  'V14', 'V15', 'V16', 'V17', 'V18', 'V19',
                                  'V20', 'V21', 'V22', 'V23', 'V24', 'V25',
                                  'V26', 'V27', 'V28', 'Hour_sin', ...])])


In [71]:
print("Input features:")
print(preprocessor.feature_names_in_)

Input features:
['Time' 'V1' 'V2' 'V3' 'V4' 'V5' 'V6' 'V7' 'V8' 'V9' 'V10' 'V11' 'V12'
 'V13' 'V14' 'V15' 'V16' 'V17' 'V18' 'V19' 'V20' 'V21' 'V22' 'V23' 'V24'
 'V25' 'V26' 'V27' 'V28' 'Amount' 'Hour_sin' 'Hour_cos']


In [72]:
print("\nNumber of input features:")
print(len(preprocessor.feature_names_in_))


Number of input features:
32


In [73]:
print("Output features:")
print(preprocessor.get_feature_names_out())

Output features:
['amount__Amount' 'numeric__Time' 'numeric__V1' 'numeric__V2'
 'numeric__V3' 'numeric__V4' 'numeric__V5' 'numeric__V6' 'numeric__V7'
 'numeric__V8' 'numeric__V9' 'numeric__V10' 'numeric__V11' 'numeric__V12'
 'numeric__V13' 'numeric__V14' 'numeric__V15' 'numeric__V16'
 'numeric__V17' 'numeric__V18' 'numeric__V19' 'numeric__V20'
 'numeric__V21' 'numeric__V22' 'numeric__V23' 'numeric__V24'
 'numeric__V25' 'numeric__V26' 'numeric__V27' 'numeric__V28'
 'numeric__Hour_sin' 'numeric__Hour_cos']


In [74]:
print("\nNumber of output features:")
print(len(preprocessor.get_feature_names_out()))


Number of output features:
32


In [75]:
xgb_model = pipeline.named_steps["model"]

print("XGBoost feature count:")
print(xgb_model.n_features_in_)

XGBoost feature count:
32


In [76]:
print("\nXGBoost feature names:")
print(xgb_model.feature_names_in_)


XGBoost feature names:
['Amount' 'Time' 'V1' 'V2' 'V3' 'V4' 'V5' 'V6' 'V7' 'V8' 'V9' 'V10' 'V11'
 'V12' 'V13' 'V14' 'V15' 'V16' 'V17' 'V18' 'V19' 'V20' 'V21' 'V22' 'V23'
 'V24' 'V25' 'V26' 'V27' 'V28' 'Hour_sin' 'Hour_cos']


In [77]:
print("Pipeline steps:")
print(pipeline.named_steps.keys())

print("\nSMOTE:")
print(pipeline.named_steps["smote"])

print("\nModel:")
print(pipeline.named_steps["model"])

Pipeline steps:
dict_keys(['smote', 'model'])

SMOTE:
SMOTE(random_state=42)

Model:
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)


In [79]:
from pathlib import Path

for path in Path("../Data/Processed").iterdir():
    print(path.name)

creditcard_clean.csv
X_test_processed.csv
X_train_processed.csv
X_train_smote.csv
y_test.csv
y_train_smote.csv
